In [144]:
import os
from warnings import filters
import jax.numpy as jnp
import jax
from caskade import Param, forward
import numpy as np
from cosmographi.cosmology import Cosmology
from cosmographi.source.base import TransientSource
from cosmographi.utils import flux
from cosmographi.utils.constants import Mpc_to_cm
from typing import Any

# V. 1.0: No parameters
The following code shows the use of an AGN source that has a constant luminosity density for testing purposes.

In [145]:
class AGNSourcev1_0(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    """
    cosmology: Cosmology
    name: str

    def __init__(self, cosmology: Cosmology = None, name: str = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
    
    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), 0.04)
        return jnp.concatenate((t, density), axis=1)
        


In [146]:
agn = AGNSourcev1_0(name="AGN1")
print(agn.luminosity_density(jnp.array([0.0, 0.5, 3.0, 5.0])))

[[0.   0.04]
 [0.5  0.04]
 [3.   0.04]
 [5.   0.04]]


# V.1.1: One parameter, luminosity as a function of time and said parameter
The following code shows an AGN whose luminosity is a function of time and a parameter of the AGN

In [147]:
from astropy.time import Time

class AGNSourcev1_1(TransientSource):
    """
    An AGN source model.

    Note: This is a simplified model that should not be used yet. 

    Parameters
    ---------- 
    name: str. 
        Optional. Name of the source.
    t0: 
        Param. Date of the initial obsevation in MJD format.
    lum_at_t0:
        Param. Luminosity density at t0 in erg/s/Hz.

    """
    cosmology: Cosmology
    name: str
    t0: Time
    lum_at_t0: Param

    def __init__(self, cosmology: Cosmology = None, name: str = None, t0: float = None, lum_at_t0: float = None, **kwargs) -> None:
        super().__init__(cosmology=cosmology, name=name, **kwargs)
        self.t0 = Param("t0", t0, shape=(), description="Date of the initial observation in MJD format")
        self.lum_at_t0 = Param("lum_at_t0", lum_at_t0, shape=(), description="Luminosity density at t0 in erg/s/Hz")

    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        """
        Compute the luminosity density at time t. 

        Parameters
        ----------
        t: jnp.ndarray. Time array in seconds. Either (N,) or (N,1)

        Returns
        -------
        luminosity_density: jnp.ndarray. Luminosity density in erg/s/Hz.
        """
        if t.ndim == 1:
            t = t[:, None]
        density = jnp.full((len(t),1), (self.lum_at_t0.value - 0.01 * (t - self.t0.value)))
        return jnp.concatenate((t, density), axis=1)

In [148]:
agn = AGNSourcev1_1(name="AGN2", t0=Time("2023-01-01T00:00:00").mjd, lum_at_t0=0.04)
print(agn.luminosity_density(jnp.array([Time("2023-01-01T00:00:00").mjd, Time("2023-01-01T00:00:00").mjd + 0.5, Time("2023-01-01T00:00:00").mjd + 3.0, Time("2023-01-01T00:00:00").mjd + 5.0])))

[[ 5.99450e+04  4.00000e-02]
 [ 5.99455e+04  3.50000e-02]
 [ 5.99480e+04  1.00000e-02]
 [ 5.99500e+04 -1.00000e-02]]


# V.1.2: One parameter, but infered through interpolation
Same as above, but the parameter is infered from some data points, which results in us using more of Caskade's functionality. 
The code below is used to generate the samples:

In [149]:
t_i = Time("2026-01-01T00:00:00").mjd
sample_times = jnp.array([t_i + 0.0, t_i + 0.5, t_i + 1.0, t_i + 1.5, t_i + 2.0, t_i + 2.5, t_i + 3.0, t_i + 3.5, t_i + 4.0, t_i + 4.5, 
                         t_i + 5.0, t_i + 5.5, t_i + 6.0])
true_lum_at_t0 = 0.04
samples = []
for time in sample_times:
    samples.append(np.random.normal(loc=true_lum_at_t0 - 0.01 * (time - t_i), scale = 0.005))
samples_jnp = jnp.array(samples)

In [150]:
class AGNSourcev1_2(AGNSourcev1_1):
    """ Includes function to ifner parameter"""

    def infer_linear_parameter(self, samples: jnp.ndarray, sample_times: jnp.ndarray) -> float:
        """
        Infer the parameter lum_at_t0 from the samples. 

        Parameters
        ----------
        samples: jnp.ndarray. Array of luminosity density samples in erg/s/Hz.
        sample_times: jnp.ndarray. Array of sample times in MJD format.

        Returns
        -------
        inferred_lum_at_t0: float. Inferred value of lum_at_t0.
        """
        slope_removed = samples - 0.01 * (sample_times - self.t0.value)
        return jnp.mean(slope_removed)

In [151]:
two_dim_samples_jnp = samples_jnp[:, None]
two_dim_sample_times = sample_times[:, None]
agn = AGNSourcev1_2(name="AGN3", t0=t_i)
agn.lum_at_t0.value = agn.infer_linear_parameter(two_dim_samples_jnp, two_dim_sample_times)
print(agn.luminosity_density(jnp.array([t_i, t_i + 0.5, t_i + 3.0, t_i + 5.0])))

[[ 6.10410000e+04 -2.16165094e-02]
 [ 6.10415000e+04 -2.66165094e-02]
 [ 6.10440000e+04 -5.16165094e-02]
 [ 6.10460000e+04 -7.16165094e-02]]


# V.1.3.: Two parameters, non-linear
Same as above, but the equation must be solved in a different way because the equation is not linear. This time, the AGN's attribute lum_at_t0 is used to calculate the shift on the x-axis of the sine function

Luminosity = a * sin[b*t-c] + d, where a and d are fixed and are assumed to be the same for all AGNs.

In [152]:

a = 0.02
d = 0.04
true_b_value = 0.25
true_c_value = t_i - 0.6
non_lin_samples = []
for time in sample_times:
    non_lin_samples.append(np.random.normal(loc=a*np.sin(true_b_value * time - true_c_value) + d, scale = 0.01))
samples_jnp_non_lin = jnp.array(non_lin_samples)



In [153]:
from scipy.optimize import curve_fit

def sin_fnctin(delta_t: float, b: float, c: float) -> float:
    """
    Return luminosity per time. the callable we will pass to curve_fit. 
    """
    return a*np.sin(b * delta_t - c) + d



def infer_b_n_c(samples: jnp.ndarray, sample_times: jnp.ndarray) -> float:
    """
    Infer the parameters b and c from the samples.

    Parameters
    ----------
    samples: jnp.ndarray. Array of luminosity density samples in erg/s/Hz.
    sample_times: jnp.ndarray. Array of sample times in MJD format.

    Returns
    -------
    inferred_b: float. Inferred value of b.
    inferred_c: float. Inferred value of c.
    """
    (inferred_b, inferred_c) = curve_fit(sin_fnctin, sample_times, samples)[0]
    print("approximate b:", inferred_b)
    print("approximate c:", inferred_c)
    return (inferred_b, inferred_c)

In [154]:
(infered_b, infered_c) = infer_b_n_c(samples_jnp_non_lin, sample_times)
inferred_lum_at_t0 = sin_fnctin(t_i, infered_b, infered_c)
from math import pi

class AGNSourcev1_3(AGNSourcev1_0):
    """Adds period parameter to the AGN source model and predicts sinusoidal luminosity density"""
    def __init__(self, name: str = None, t0: float = None, lum_at_t0: float = None, period: float = None, c: float = None, **kwargs) -> None:
        super().__init__(name=name, t0=t0, **kwargs)
        self.period = Param("period", period, shape=(), description="Period of the sinusoidal variation")
        self.c = Param("c", c, shape=(), description="Horizontal shift of the sinusoidal variation")

    def luminosity_density(self, t: jnp.ndarray) -> jnp.ndarray:
        return sin_fnctin(t, pi / self.period.value, self.c.value)


agn = AGNSourcev1_3(name="AGN4", period=pi / infered_b, c=infered_c)
print(agn.luminosity_density(jnp.array([t_i, t_i + 0.5, t_i + 3.0, t_i + 5.0])))

approximate b: 1.124005101377442
approximate c: 7572.486045521535
[0.04188358 0.0309836  0.04271368 0.05374184]
